# 04 · Validación de Datos
Verifica integridad, conteos y calidad básica de cada tabla.

In [2]:
from google.cloud import bigquery
from dotenv import load_dotenv
import os, pandas as pd

load_dotenv()
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')
PROJECT_ID = os.getenv('GCP_PROJECT_ID')
DATASET_ID = os.getenv('BQ_DATASET_ID')
client = bigquery.Client(project=PROJECT_ID)
DS = f'{PROJECT_ID}.{DATASET_ID}'

In [3]:
tables = ['categories','customers','products','orders','order_item_id','payments','reviews']
rows = []
for t in tables:
    n = client.query(f'SELECT COUNT(*) as n FROM `{DS}.{t}`').result().to_dataframe().iloc[0]['n']
    rows.append({'tabla': t, 'filas': n})
pd.DataFrame(rows)

c:\Users\Usuario\Documents\GitHub\Repositorio_persona\TC-Novarix-Data\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
c:\Users\Usuario\Documents\GitHub\Repositorio_persona\TC-Novarix-Data\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
c:\Users\Usuario\Documents\GitHub\Repositorio_persona\TC-Novarix-Data\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
c:\Users\Usuario\Documents\GitHub\Repositorio_persona\TC-Novarix-Data\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
c:\Users\Usuario\Documents\GitHub\Repositorio_pe

,tabla,filas
0,categories,10
1,customers,500
2,products,150
3,orders,2000
4,order_item_id,5995
5,payments,2000
6,reviews,1379


In [4]:
q = f'SELECT status, COUNT(*) AS total FROM `{DS}.orders` GROUP BY status ORDER BY total DESC'
client.query(q).result().to_dataframe()

c:\Users\Usuario\Documents\GitHub\Repositorio_persona\TC-Novarix-Data\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,status,total
0,entregado,1204
1,enviado,306
2,en_proceso,197
3,cancelado,139
4,pendiente,96
5,devuelto,58


In [5]:
q = f"""
SELECT COUNT(DISTINCT o.order_id) AS num_orders,
       ROUND(SUM(oi.unit_price * oi.quantity * (1 - oi.discount)), 2) AS revenue_total,
       ROUND(AVG(oi.unit_price * oi.quantity * (1 - oi.discount)), 2) AS avg_item_value
FROM `{DS}.orders` o JOIN `{DS}.order_item_id` oi USING (order_id)
WHERE o.status = 'entregado'"""
client.query(q).result().to_dataframe()

c:\Users\Usuario\Documents\GitHub\Repositorio_persona\TC-Novarix-Data\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,num_orders,revenue_total,avg_item_value
0,1204,462830.05,128.35


In [6]:
q = f"""
SELECT p.name, SUM(oi.quantity) AS units_sold,
       ROUND(SUM(oi.unit_price * oi.quantity * (1-oi.discount)),2) AS revenue
FROM `{DS}.order_item_id` oi JOIN `{DS}.products` p USING (product_id)
JOIN `{DS}.orders` o USING (order_id)
WHERE o.status = 'entregado'
GROUP BY p.name ORDER BY revenue DESC LIMIT 5"""
client.query(q).result().to_dataframe()

c:\Users\Usuario\Documents\GitHub\Repositorio_persona\TC-Novarix-Data\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,name,units_sold,revenue
0,"Monitor LED 27"" - Azul Marino",70,14662.96
1,"Monitor LED 27"" - Gris",59,14044.28
2,"Monitor LED 27"" - Beige",56,13067.09
3,"Monitor LED 27""",58,11994.40
4,Perfume EDP 50ml - Negro,156,11331.94


In [7]:
q = f'SELECT rating, COUNT(*) AS total FROM `{DS}.reviews` GROUP BY rating ORDER BY rating DESC'
client.query(q).result().to_dataframe()

c:\Users\Usuario\Documents\GitHub\Repositorio_persona\TC-Novarix-Data\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,rating,total
0,5,650
1,4,409
2,3,157
3,2,106
4,1,57


In [8]:
q = f'SELECT acquisition_channel, COUNT(*) AS clientes FROM `{DS}.customers` GROUP BY 1 ORDER BY 2 DESC'
client.query(q).result().to_dataframe()

c:\Users\Usuario\Documents\GitHub\Repositorio_persona\TC-Novarix-Data\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,acquisition_channel,clientes
0,redes_sociales,93
1,correo_electrónico,85
2,búsqueda_pagada,83
3,directo,82
4,referencia,79
5,orgánico,78


In [8]:
q = f'SELECT method, status, COUNT(*) AS n, ROUND(SUM(amount),2) AS total_amount FROM `{DS}.payments` GROUP BY 1,2 ORDER BY 1,3 DESC'
client.query(q).result().to_dataframe()

c:\Users\Usuario\Documents\GitHub\Repositorio_persona\TC-Novarix-Data\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,method,status,n,total_amount
0,criptomonedas,completado,302,117323.32
1,criptomonedas,pendiente,45,17720.53
2,criptomonedas,reembolsado,26,9914.89
3,criptomonedas,fallido,25,9112.17
4,paypal,completado,319,120124.35
5,paypal,pendiente,42,16840.49
6,paypal,fallido,22,11044.42
7,paypal,reembolsado,15,6077.41
8,tarjeta_de_crédito,completado,330,130619.68
9,tarjeta_de_crédito,pendiente,35,14698.30
